# [실습 2] Datasets 전처리와 커스텀 데이터셋 구성


## 실습 개요

> "로컬 데이터도 Hugging Face Datasets 형식으로 다루면 어떤 장점이 있을까?"

이번 실습에서는 `datasets` 라이브러리로 로컬 데이터를 만들고, `DatasetDict`, `map()`, `filter()`, `select()`, `sort()`를 사용합니다.  
Apache Arrow 기반 Dataset의 컬럼 중심 처리와 Trainer 연동 전처리 흐름을 작은 예제로 익힙니다.


## 실습 목표

1. pandas DataFrame을 `Dataset`으로 변환한다.
2. train/validation 구조의 `DatasetDict`를 만든다.
3. `map()`, `filter()`, `select()`, `sort()` 핵심 연산을 사용한다.
4. `Features`로 컬럼 스키마를 확인한다.
5. [TODO] 토큰화와 길이 컬럼 생성을 수행한다.


## 데이터 출처

- 데이터: 실습용 텍스트/라벨 직접 구성
- 라이브러리: Hugging Face `datasets`, `transformers`


## 목차

1. 라이브러리 임포트
2. Dataset과 DatasetDict 생성
3. 핵심 연산 map/filter/select/sort
4. Features와 컬럼 확인
5. [TODO] 토큰화 전처리 함수 작성


---
## 1. 라이브러리 임포트


### Datasets 실습 환경 준비

로컬 데이터를 Hugging Face 학습 파이프라인에 맞춰 다루려면 `Dataset`, `DatasetDict`, `Features`, `ClassLabel`의 역할을 구분해야 합니다. 데이터 저장 구조, split 구성, 컬럼 타입, 라벨 매핑을 명확히 관리하기 위한 기본 객체들입니다.

토크나이저는 뒤에서 텍스트 컬럼을 모델 입력 컬럼으로 변환할 때 사용합니다. 데이터셋 객체와 토크나이저가 함께 움직인다는 점을 잡고 보면 이후 전처리 흐름을 이해하기 쉽습니다.

In [ ]:
%pip install datasets

In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
from transformers import AutoTokenizer

model_name = 'HuggingFaceTB/SmolLM2-135M-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
print('준비 완료')

준비 완료


---
## 2. Dataset과 DatasetDict 생성


### 로컬 데이터를 DatasetDict로 구성하기

작은 pandas DataFrame도 `Dataset`으로 변환하면 Hugging Face 생태계의 `map()`, `filter()`, Trainer 연동 기능을 그대로 사용할 수 있습니다. CSV, JSON, 데이터베이스에서 가져온 데이터를 학습용 파이프라인에 올릴 때도 같은 흐름으로 정리합니다.

`DatasetDict`는 train, validation, test 같은 split을 한 객체에서 관리하게 해 줍니다. 이렇게 구성해 두면 전처리 함수를 각 split에 일관되게 적용하거나 Trainer에 그대로 전달하기가 편합니다.

`Features`와 `ClassLabel`은 컬럼의 의미와 타입을 고정하는 장치입니다. 특히 문자열 라벨을 정수 ID로 안정적으로 매핑해 주기 때문에 전처리, 학습, 평가 단계에서 라벨 해석이 흔들리지 않습니다.

In [ ]:
raw_rows = [
    {'text': 'Pipeline API makes inference simple.', 'label': 'positive'},
    {'text': 'The preprocessing step is too slow.', 'label': 'negative'},
    {'text': 'Datasets can map functions efficiently.', 'label': 'positive'},
    {'text': 'Streaming is useful for large data.', 'label': 'positive'},
    {'text': 'Bad schema design causes confusion.', 'label': 'negative'},
    {'text': 'Trainer expects tokenized columns.', 'label': 'positive'},
]

df = pd.DataFrame(raw_rows)
features = Features({
    'text': Value('string'),
    'label': ClassLabel(names=['negative', 'positive']),
})

dataset = Dataset.from_pandas(df).cast(features)

dataset_dict = DatasetDict({
    'train': dataset.select(range(4)),
    'validation': dataset.select(range(4, 6)),
})

print(dataset_dict)
print(dataset_dict['train'][0])

Casting the dataset: 100%|██████████| 6/6 [00:00<00:00, 1071.07 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 4
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2
    })
})
{'text': 'Pipeline API makes inference simple.', 'label': 1}


---
## 3. 핵심 연산 map/filter/select/sort @@@@


### map, filter, select, sort로 데이터 가공하기

Datasets에서 가장 자주 쓰는 연산은 `map()`, `filter()`, `select()`, `sort()`입니다. `map()`은 새 컬럼을 만들거나 값을 변환할 때, `filter()`는 조건에 맞는 샘플만 남길 때 사용합니다.

`select()`는 특정 인덱스만 뽑아 빠르게 실험하거나 일부 샘플을 확인할 때 유용합니다. `sort()`는 길이, 점수, 날짜 같은 컬럼 기준으로 샘플 순서를 바꿀 때 사용합니다.

대규모 데이터에서는 이런 연산이 Arrow 기반으로 효율적으로 처리됩니다. 토크나이징, 품질 필터링, 샘플링, 정렬을 단계별 파이프라인으로 연결하면 원본 데이터와 전처리 결과를 비교적 깔끔하게 관리할 수 있습니다.

In [ ]:
def add_text_length(example):
    example['text_length'] = len(example['text'])
    return example

with_length = dataset_dict.map(add_text_length)
positive_only = with_length['train'].filter(lambda x: x['label'] == 1)
short_subset = with_length['train'].select([0, 2])
sorted_by_length = with_length['train'].sort('text_length')

print('positive_only:', len(positive_only))
print('short_subset:', short_subset[:])
print('sorted text_length:', sorted_by_length['text_length'])

Filter: 100%|██████████| 4/4 [00:00<00:00, 254.17 examples/s]

positive_only: 3
short_subset: {'text': ['Pipeline API makes inference simple.', 'Datasets can map functions efficiently.'], 'label': [1, 1], 'text_length': [36, 39]}
sorted text_length: Column([35, 35, 36, 39])


---
## 4. Features와 컬럼 확인


### 스키마와 컬럼 구조 확인

전처리를 적용한 뒤에는 `features`, `column_names`, 실제 샘플을 확인해야 합니다. 컬럼 이름이나 타입이 모델이 기대하는 입력과 다르면 학습 단계에서 오류가 발생합니다.

텍스트 분류 학습에서는 보통 `input_ids`, `attention_mask`, `labels` 같은 컬럼이 필요합니다. 현재 데이터셋이 모델 입력으로 넘어갈 준비가 되었는지 점검하는 짧은 확인 단계입니다.

In [6]:
print(with_length['train'].features)
print(with_length['train'].column_names)
print(with_length['train'][:2])

{'text': Value('string'), 'label': ClassLabel(names=['negative', 'positive']), 'text_length': Value('int64')}
['text', 'label', 'text_length']
{'text': ['Pipeline API makes inference simple.', 'The preprocessing step is too slow.'], 'label': [1, 0], 'text_length': [36, 35]}


---
### [TODO] 토큰화 전처리 함수 작성 !!!!!!!!!!!!!

`tokenize_batch()` 함수를 완성해 Dataset에 토큰화 컬럼을 추가하세요.

구현 조건:
- 입력 인자는 batch 딕셔너리입니다.
- `batch['text']`를 토크나이징합니다.
- `padding='max_length'`, `truncation=True`, `max_length=32`를 사용합니다.
- `token_length` 컬럼에는 각 샘플의 실제 토큰 수를 저장합니다.


### batched map으로 토큰화 전처리하기

`batched=True`로 `map()`을 실행하면 전처리 함수가 샘플 하나가 아니라 여러 샘플을 묶은 딕셔너리를 받습니다. 토크나이저도 리스트 입력을 한 번에 처리하므로, 실제 데이터가 많을 때 훨씬 자연스럽고 효율적인 전처리 방식입니다.

`padding='max_length'`는 모든 샘플을 같은 길이로 맞추고, `truncation=True`는 긴 텍스트를 지정한 길이에 맞게 자릅니다. 이 설정은 모델 입력 shape을 고정하고 싶을 때 편하지만, 너무 큰 max length를 잡으면 불필요한 padding이 늘어날 수 있습니다.

`token_length`는 attention mask에서 실제 토큰 개수를 세어 만든 보조 컬럼입니다. 입력 길이 분포를 분석하거나, 너무 긴 샘플을 찾거나, 길이에 따른 성능 차이를 살펴볼 때 이런 파생 컬럼이 도움이 됩니다.

In [ ]:
def tokenize_batch(batch):
    tokenized = tokenizer(
        batch['text'], 
        padding='max_length', 
        truncation=True, max_length=32
        )
    tokenized['token_length'] = [len(tokens) for tokens in tokenized['input_ids']]
    return tokenized

tokenized_dataset = with_length.map(tokenize_batch, batched=True)
tokenized_dataset

Map: 100%|██████████| 2/2 [00:00<00:00, 645.53 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'text_length', 'input_ids', 'attention_mask', 'token_length'],
        num_rows: 4
    })
    validation: Dataset({
        features: ['text', 'label', 'text_length', 'input_ids', 'attention_mask', 'token_length'],
        num_rows: 2
    })
})

### 토큰화 결과 살펴보기

전처리 함수가 실행된 뒤에는 새 컬럼이 의도대로 추가되었는지 확인합니다. 첫 번째 샘플에 `input_ids`, `attention_mask`, `token_length`가 들어 있는지 보면 토큰화 결과를 빠르게 점검할 수 있습니다.

전처리 코드는 오류 없이 실행되어도 컬럼명이 잘못되거나 예상과 다른 타입을 만들 수 있습니다. 학습에 넘기기 전에 샘플 하나를 직접 출력해 보는 습관이 좋습니다.

In [11]:
print(tokenized_dataset['train'].column_names)
print(tokenized_dataset['train'][0])

['text', 'label', 'text_length', 'input_ids', 'attention_mask', 'token_length']
{'text': 'Pipeline API makes inference simple.', 'label': 1, 'text_length': 36, 'input_ids': [42914, 4374, 12077, 2022, 23630, 2232, 30, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_length': 32}
